In [1]:
from openai import OpenAI
import json

ModuleNotFoundError: No module named 'openai'

In [2]:
!pip install openai


  Obtaining dependency information for openai from https://files.pythonhosted.org/packages/c9/30/844dc675ee6902579b8eef01ed23917cc9319a1c9c0c14ec6e39340c96d0/openai-2.24.0-py3-none-any.whl.metadata
  Obtaining dependency information for distro<2,>=1.7.0 from https://files.pythonhosted.org/packages/12/b3/231ffd4ab1fc9d679809f356cebee130ac7daa00d6d6f3206dd4fd137e9e/distro-1.9.0-py3-none-any.whl.metadata
  Obtaining dependency information for httpx<1,>=0.23.0 from https://files.pythonhosted.org/packages/2a/39/e50c7c3a983047577ee07d2a9e53faf5a69493943ec3f6a384bdc792deb2/httpx-0.28.1-py3-none-any.whl.metadata
  Obtaining dependency information for jiter<1,>=0.10.0 from https://files.pythonhosted.org/packages/cb/16/8e8203ce92f844dfcd3d9d6a5a7322c77077248dbb12da52d23193a839cd/jiter-0.13.0-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for httpcore==1.* from https://files.pythonhosted.org/packages/7e/f5/f66802a942d491edb555dd61e3a9961140fd64c90bce1eafd741609d334d/httpcor

In [3]:
import openai
print(openai.__version__)

2.24.0


In [4]:
from openai import OpenAI
import json

In [5]:
client = OpenAI(api_key="your_own_api_key")


In [6]:
MAX_ITERATIONS = 5
MAX_WORDS = 120

In [7]:
def generate_summary(text, feedback=None):
    system_prompt = (
        "You are a summarization agent. "
        "Produce a clear, beginner-friendly summary under 120 words. "
        "Avoid technical jargon. "
        "Apply feedback if provided."
    )
    
    user_prompt = f"TEXT:\n{text}\n"
    
    if feedback:
        user_prompt += f"\nFEEDBACK:\n{feedback}"
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    
    return response.choices[0].message.content.strip()

In [8]:
def evaluate_summary(summary):
    word_count = len(summary.split())
    
    system_prompt = (
        "You are a strict evaluator. "
        "Evaluate the summary against these criteria:\n"
        "- Beginner-friendly language\n"
        "- Clear and concise\n"
        "- No technical jargon\n\n"
        "Do NOT rewrite the summary.\n"
        "Return ONLY valid JSON in this format:\n"
        "{\n"
        '  "beginner_friendly": true/false,\n'
        '  "clarity": true/false,\n'
        '  "jargon_free": true/false,\n'
        '  "issues": [string],\n'
        '  "suggestions": [string]\n'
        "}"
    )
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": summary}
        ]
    )
    
    eval_data = json.loads(response.choices[0].message.content)
    
    eval_data["word_count_ok"] = word_count <= MAX_WORDS
    eval_data["word_count"] = word_count
    
    eval_data["passed"] = (
        eval_data["word_count_ok"]
        and eval_data["beginner_friendly"]
        and eval_data["clarity"]
        and eval_data["jargon_free"]
    )
    
    return eval_data

In [9]:
def run_agent(text):
    feedback = None
    best_summary = None
    
    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"\n--- Iteration {iteration} ---")
        
        summary = generate_summary(text, feedback)
        evaluation = evaluate_summary(summary)
        
        print(f"Word count: {evaluation['word_count']}")
        print("Passed:", evaluation["passed"])
        
        if evaluation["passed"]:
            print("✅ Goal achieved. Stopping.")
            return summary
        
        feedback = "\n".join(evaluation["suggestions"])
        best_summary = summary
        
        print("🔁 Refining based on feedback...")
    
    print("⚠️ Max iterations reached. Returning best attempt.")
    return best_summary

In [11]:
input_text = """
AI agents are systems that use artificial intelligence to perform tasks
autonomously by perceiving their environment, making decisions, and taking
actions to achieve specific goals. They are often powered by large language
models and can use tools, memory, and planning strategies.
"""

final_summary = run_agent(input_text)
print("\nFINAL SUMMARY:\n", final_summary)


--- Iteration 1 ---
Word count: 48
Passed: False
🔁 Refining based on feedback...

--- Iteration 2 ---
Word count: 43
Passed: True
✅ Goal achieved. Stopping.

FINAL SUMMARY:
 AI agents are computer programs that use artificial intelligence to work on their own. They observe their surroundings, make choices, and take actions to reach specific goals. These agents can gather information, remember past events, and plan their activities to complete tasks efficiently.
